# 1. Data Load

In [187]:
import os
from pathlib import Path

In [188]:
data_path = Path("../data")

movie_path = f"{data_path}/movies.csv"
rating_path = f"{data_path}/ratings.csv"
tag_path = f"{data_path}/tags.csv"
link_path = f"{data_path}/links.csv"

In [189]:
import pandas as pd

movies_df = pd.read_csv(movie_path)
ratings_df = pd.read_csv(rating_path)
tags_df = pd.read_csv(tag_path)
links_df = pd.read_csv(link_path)


## a) Null check

In [190]:
df_list = [movies_df, ratings_df, tags_df]

for df in df_list:
    print(df.isnull().sum(), '\n')

movieId    0
title      0
genres     0
dtype: int64 

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64 

userId       0
movieId      0
tag          0
timestamp    0
dtype: int64 



# b) index check

In [191]:
movies_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB


In [192]:
ratings_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [193]:
tags_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3683 entries, 0 to 3682
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   userId     3683 non-null   int64 
 1   movieId    3683 non-null   int64 
 2   tag        3683 non-null   object
 3   timestamp  3683 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 115.2+ KB


movie id, user id 는 속도를 위해 int 형으로 그대로 사용했습니다.

# c) check duplicate

In [194]:
for col in movies_df:
    print(f"{col} : \t", movies_df[col].nunique())

movieId : 	 9742
title : 	 9737
genres : 	 951


In [195]:
duplicated_title = movies_df[movies_df['title'].duplicated()]['title']
movies_df[movies_df['title'].duplicated()]

,movieId,title,genres
5601,26958,Emma (1996),Romance
6932,64997,War of the Worlds (2005),Action|Sci-Fi
9106,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller
9135,147002,Eros (2004),Drama|Romance
9468,168358,Saturn 3 (1980),Sci-Fi|Thriller


In [196]:
dup_temp = movies_df[movies_df['title'].isin(duplicated_title)].sort_values(by='title')
dup_temp

,movieId,title,genres
4169,6003,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Thriller
9106,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller
650,838,Emma (1996),Comedy|Drama|Romance
5601,26958,Emma (1996),Romance
5854,32600,Eros (2004),Drama
9135,147002,Eros (2004),Drama|Romance
2141,2851,Saturn 3 (1980),Adventure|Sci-Fi|Thriller
9468,168358,Saturn 3 (1980),Sci-Fi|Thriller
5931,34048,War of the Worlds (2005),Action|Adventure|Sci-Fi|Thriller
6932,64997,War of the Worlds (2005),Action|Sci-Fi


In [197]:
# genres 가 가장 긴 영화 기준으로 중복 데이터 처리
dup_temp['genres_len'] = dup_temp['genres'].str.len()
dup_temp

,movieId,title,genres,genres_len
4169,6003,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Thriller,27
9106,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller,35
650,838,Emma (1996),Comedy|Drama|Romance,20
5601,26958,Emma (1996),Romance,7
5854,32600,Eros (2004),Drama,5
9135,147002,Eros (2004),Drama|Romance,13
2141,2851,Saturn 3 (1980),Adventure|Sci-Fi|Thriller,25
9468,168358,Saturn 3 (1980),Sci-Fi|Thriller,15
5931,34048,War of the Worlds (2005),Action|Adventure|Sci-Fi|Thriller,32
6932,64997,War of the Worlds (2005),Action|Sci-Fi,13


In [198]:
group_temp = dup_temp.groupby(by='title')
dup_title_max_len_idx = group_temp['genres_len'].idxmax() # 데이터 검산을 위한 drop idx 와 max_len_idx → 데이터 삭제 이후에는 인덱스 접근이 아닌 키값으로 접근할 것

movies_df.loc[dup_title_max_len_idx]

,movieId,title,genres
9106,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller
650,838,Emma (1996),Comedy|Drama|Romance
9135,147002,Eros (2004),Drama|Romance
2141,2851,Saturn 3 (1980),Adventure|Sci-Fi|Thriller
5931,34048,War of the Worlds (2005),Action|Adventure|Sci-Fi|Thriller


In [199]:
drop_idx = dup_temp.index.difference(dup_title_max_len_idx)
dup_temp.loc[drop_idx]

,movieId,title,genres,genres_len
4169,6003,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Thriller,27
5601,26958,Emma (1996),Romance,7
5854,32600,Eros (2004),Drama,5
6932,64997,War of the Worlds (2005),Action|Sci-Fi,13
9468,168358,Saturn 3 (1980),Sci-Fi|Thriller,15


In [200]:
drop_movieId = dup_temp.loc[drop_idx]['movieId']
max_len_movieId = dup_temp.loc[dup_title_max_len_idx]['movieId']

In [201]:
movies_refine = movies_df.drop(drop_idx).copy()

In [202]:
print("삭제된 ROW :\t", abs(len(movies_df) - len(movies_refine)))
print("dup index 갯수 :\t", len(drop_idx))

삭제된 ROW :	 5
dup index 갯수 :	 5


In [203]:
movies_refine.loc[dup_title_max_len_idx]

,movieId,title,genres
9106,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller
650,838,Emma (1996),Comedy|Drama|Romance
9135,147002,Eros (2004),Drama|Romance
2141,2851,Saturn 3 (1980),Adventure|Sci-Fi|Thriller
5931,34048,War of the Worlds (2005),Action|Adventure|Sci-Fi|Thriller


In [204]:
max_len_movieId

9106    144606
650        838
9135    147002
2141      2851
5931     34048
Name: movieId, dtype: int64

In [205]:
# 중복 제거 확인용
movies_refine[movies_refine['movieId'].isin(max_len_movieId.values)]

,movieId,title,genres
650,838,Emma (1996),Comedy|Drama|Romance
2141,2851,Saturn 3 (1980),Adventure|Sci-Fi|Thriller
5931,34048,War of the Worlds (2005),Action|Adventure|Sci-Fi|Thriller
9106,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller
9135,147002,Eros (2004),Drama|Romance


In [206]:
movies_refine[movies_refine['movieId'].isin(drop_movieId.values)]

,movieId,title,genres


In [207]:
movies_df = movies_refine

In [208]:
max_len_movieId.values

array([144606,    838, 147002,   2851,  34048])

In [209]:
idx_list = group_temp['movieId'].apply(list)
idx_list

title
Confessions of a Dangerous Mind (2002)     [6003, 144606]
Emma (1996)                                  [838, 26958]
Eros (2004)                               [32600, 147002]
Saturn 3 (1980)                            [2851, 168358]
War of the Worlds (2005)                   [34048, 64997]
Name: movieId, dtype: object

In [210]:
# mapping 을 위한 series 생성
map_series = dup_temp.loc[dup_title_max_len_idx].set_index('title')['movieId']
map_series

title
Confessions of a Dangerous Mind (2002)    144606
Emma (1996)                                  838
Eros (2004)                               147002
Saturn 3 (1980)                             2851
War of the Worlds (2005)                   34048
Name: movieId, dtype: int64

In [211]:
mapping_dict = {}
for title, movie_ids in idx_list.items():
    dict_value = map_series[title]
    for id in movie_ids:
        mapping_dict[id] = dict_value

In [212]:
mapping_dict

{6003: 144606,
 144606: 144606,
 838: 838,
 26958: 838,
 32600: 147002,
 147002: 147002,
 2851: 2851,
 168358: 2851,
 34048: 34048,
 64997: 34048}

In [213]:
# 중복치 제거 확인용
ratings_df[ratings_df['movieId'].isin(drop_movieId)]

,userId,movieId,rating,timestamp
4747,28,64997,3.5,1234850075
11451,68,64997,2.5,1230497715
17449,111,6003,4.0,1516468531
23053,156,6003,3.5,1106882187
26958,182,6003,3.0,1054780821
42984,288,6003,4.0,1066059244
54020,356,6003,4.5,1229139513
59953,387,6003,3.5,1208707060
64063,414,6003,3.5,1092414917
74530,474,6003,3.5,1087831997


In [214]:
ratings_refine = ratings_df['movieId'].map(mapping_dict)

In [215]:
ratings_refine = ratings_refine.fillna(ratings_df['movieId'])

In [216]:
# 정제 데이터를 기존 데이터로 변경
ratings_df['movieId'] = ratings_refine

In [217]:
# 중복 제거 확인2
ratings_df[ratings_df['movieId'].isin(drop_movieId)]

,userId,movieId,rating,timestamp


In [218]:
tags_df[tags_df['movieId'].isin(drop_movieId)]

,userId,movieId,tag,timestamp
2058,474,6003,television,1138307058


In [219]:
tags_refine = tags_df['movieId'].map(mapping_dict)
tags_refine.fillna(tags_df['movieId'])
tags_df['movieId'] = tags_refine
tags_df[tags_df['movieId'].isin(drop_movieId)]

,userId,movieId,tag,timestamp


In [220]:
links_df[links_df['movieId'].isin(drop_movieId)]

,movieId,imdbId,tmdbId
4169,6003,290538,4912.0
5601,26958,118308,12254.0
5854,32600,377059,NaN
6932,64997,449040,34812.0
9468,168358,79285,19761.0


In [221]:
links_refine = links_df['movieId'].map(mapping_dict)
links_refine.fillna(links_df['movieId'])
links_df['movieId'] = tags_refine
links_df[links_df['movieId'].isin(drop_movieId)]

,movieId,imdbId,tmdbId


In [222]:
print('movie_df 의 중복치 제거 확인 (0 이면 제거 완료) :\t', movies_df['movieId'].isin(drop_movieId).sum())
print('rating_df 의 중복치 제거 확인 :\t',ratings_df['movieId'].isin(drop_movieId).sum())
print('tags_df 의 중복치 제거 확인 :\t', tags_df['movieId'].isin(drop_movieId).sum())
print('links_df 의 중복치 제거 확인 :\t', links_df['movieId'].isin(drop_movieId).sum())

movie_df 의 중복치 제거 확인 (0 이면 제거 완료) :	 0
rating_df 의 중복치 제거 확인 :	 0
tags_df 의 중복치 제거 확인 :	 0
links_df 의 중복치 제거 확인 :	 0


# 장르 결측치 처리

https://www.themoviedb.org/

## a) check missing value

In [223]:
print(movies_df['genres'].str.contains('no genres listed').sum(),"개의 장르 결측치 존재")

34 개의 장르 결측치 존재


In [224]:
genres_null = movies_df[movies_df['genres'].str.contains('no genres listed')]
genres_null

,movieId,title,genres
8517,114335,La cravate (1957),(no genres listed)
8684,122888,Ben-hur (2016),(no genres listed)
8687,122896,Pirates of the Caribbean: Dead Men Tell No Tal...,(no genres listed)
8782,129250,Superfast! (2015),(no genres listed)
8836,132084,Let It Be Me (1995),(no genres listed)
8902,134861,Trevor Noah: African American (2013),(no genres listed)
9033,141131,Guardians (2016),(no genres listed)
9053,141866,Green Room (2015),(no genres listed)
9070,142456,The Brand New Testament (2015),(no genres listed)
9091,143410,Hyena Road,(no genres listed)


In [248]:
pd.set_option('display.max_colwidth', None)
genres_null

,movieId,title,genres
8517,114335,La cravate (1957),(no genres listed)
8684,122888,Ben-hur (2016),(no genres listed)
8687,122896,Pirates of the Caribbean: Dead Men Tell No Tales (2017),(no genres listed)
8782,129250,Superfast! (2015),(no genres listed)
8836,132084,Let It Be Me (1995),(no genres listed)
8902,134861,Trevor Noah: African American (2013),(no genres listed)
9033,141131,Guardians (2016),(no genres listed)
9053,141866,Green Room (2015),(no genres listed)
9070,142456,The Brand New Testament (2015),(no genres listed)
9091,143410,Hyena Road,(no genres listed)


# b) check genres value for replacing Null

In [225]:
exist_genre_index = movies_df.index.difference(genres_null.index)
all_genres = movies_df.loc[exist_genre_index]['genres'].replace('\|', ' ', regex=True).str.split(' ').explode().unique()

<>:2: SyntaxWarning: invalid escape sequence '\|'
<>:2: SyntaxWarning: invalid escape sequence '\|'
/var/folders/1t/sts9dw9x317fjpv6tzhvwldw0000gn/T/ipykernel_70224/2543130846.py:2: SyntaxWarning: invalid escape sequence '\|'
  all_genres = movies_df.loc[exist_genre_index]['genres'].replace('\|', ' ', regex=True).str.split(' ').explode().unique()


In [226]:
all_genres

array(['Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy',
       'Romance', 'Drama', 'Action', 'Crime', 'Thriller', 'Horror',
       'Mystery', 'Sci-Fi', 'War', 'Musical', 'Documentary', 'IMAX',
       'Western', 'Film-Noir'], dtype=object)

# c) check replace

In [227]:
movies_df[movies_df['title'] == 'La cravate (1957)']

,movieId,title,genres
8517,114335,La cravate (1957),(no genres listed)


In [228]:
movies_df.loc[movies_df['title']=='La cravate (1957)', 'genres']='Fantasy Comedy'


In [229]:
movies_df[movies_df['title'] == 'La cravate (1957)']

,movieId,title,genres
8517,114335,La cravate (1957),Fantasy Comedy


In [ ]:
movies_df.loc[movies_df['title']=='Ben-hur (2016)', 'genres'] = 'Action Adventure Drama'
movies_df.loc[movies_df['title']=='Pirates of the Caribbean: Dead Men Tell No Tales (2017)', 'genres'] = 'Action Adventure Fantasy'
movies_df.loc[movies_df['title']=='Superfast! (2015)'] = 'Comedy Action'
movies_df.loc[movies_df['title']=='Let It Be Me (1995)', 'genres'] = 'Romance'
movies_df.loc[movies_df['title']=='Trevor Noah: African American (2013)', 'genres'] = 'Comedy'
movies_df.loc[movies_df['title']=='Guardians (2016)', 'genres'] = 'Action Adventure Drama'
movies_df.loc[movies_df['title']=='Ben-hur (2016)', 'genres'] = 'Action Adventure Drama'
movies_df.loc[movies_df['title']=='Ben-hur (2016)', 'genres'] = 'Action Adventure Drama'
movies_df.loc[movies_df['title']=='Ben-hur (2016)', 'genres'] = 'Action Adventure Drama'
movies_df.loc[movies_df['title']=='Ben-hur (2016)', 'genres'] = 'Action Adventure Drama'
movies_df.loc[movies_df['title']=='Ben-hur (2016)', 'genres'] = 'Action Adventure Drama'
movies_df.loc[movies_df['title']=='Ben-hur (2016)', 'genres'] = 'Action Adventure Drama'
movies_df.loc[movies_df['title']=='Ben-hur (2016)', 'genres'] = 'Action Adventure Drama'
movies_df.loc[movies_df['title']=='Ben-hur (2016)', 'genres'] = 'Action Adventure Drama'
movies_df.loc[movies_df['title']=='Ben-hur (2016)', 'genres'] = 'Action Adventure Drama'
movies_df.loc[movies_df['title']=='Ben-hur (2016)', 'genres'] = 'Action Adventure Drama'
movies_df.loc[movies_df['title']=='Ben-hur (2016)', 'genres'] = 'Action Adventure Drama'
movies_df.loc[movies_df['title']=='Ben-hur (2016)', 'genres'] = 'Action Adventure Drama'
movies_df.loc[movies_df['title']=='Ben-hur (2016)', 'genres'] = 'Action Adventure Drama'
movies_df.loc[movies_df['title']=='Ben-hur (2016)', 'genres'] = 'Action Adventure Drama'
